# 146. LRU Cache

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** design, hash-table, linked-list
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/lru-cache/)

Design a data structure that follows the constraints of a **Least Recently Used
(LRU) cache**.

Implement the `LRUCache` class:

- `LRUCache(capacity)` initialises the LRU cache with **positive** size `capacity`.
- `get(key)` returns the value of the `key` if it exists, otherwise returns `-1`.
- `put(key, value)` updates the value of the `key` if it exists. Otherwise, adds the
  `key`-`value` pair to the cache. If the number of keys exceeds the `capacity` from
  this operation, **evict the least recently used key**.

The functions `get` and `put` must each run in `O(1)` **average** time complexity.

---

### Example

```
Input:  ["LRUCache","put","put","get","put","get","put","get","get","get"]
        [[2],       [1,1],[2,2],[1],  [3,3],[2],  [4,4],[1],  [3],  [4]]
Output: [null,      null, null, 1,    null, -1,   null, -1,   3,    4]

LRUCache cache = new LRUCache(2);
cache.put(1, 1);   // cache is {1=1}
cache.put(2, 2);   // cache is {1=1, 2=2}
cache.get(1);      // returns 1        cache is {2=2, 1=1}  <- 1 is now the most recent
cache.put(3, 3);   // evicts key 2     cache is {1=1, 3=3}
cache.get(2);      // returns -1 (not found)
cache.put(4, 4);   // evicts key 1     cache is {4=4, 3=3}
cache.get(1);      // returns -1 (not found)
cache.get(3);      // returns 3
cache.get(4);      // returns 4
```

---

### Constraints

- `1 <= capacity <= 3000`
- `0 <= key <= 10^4`
- `0 <= value <= 10^5`
- At most `2 * 10^5` calls will be made to `get` and `put`

**This is the one #707 was pointing at.** Its closing note said you were one dict away
from an LRU cache, and this is the dict. Every cache you have ever used - your CPU's,
your browser's, Redis with `maxmemory-policy allkeys-lru` - is this class. It is also
the most asked design question in interviews, for the good reason that it is the
smallest problem where **one structure is not enough**.

## Before you write anything

**1.** Write the two requirements as separate sentences:

```
"find the value for this key"           in O(1)
"find the key that was used longest ago" in O(1)
```

Now name a structure that gives you the first. Name one that gives you the second.
Notice that they are **different structures** - and that the whole problem is making
one object satisfy both at once. Say what each structure would cost you if you tried
to use it alone for both jobs.

**2.** The tempting single-structure answer is a list of keys in use-order plus a dict
for the values. `get(key)` then has to move that key to the front, which means
**finding** it in the list first - `list.remove(key)` or `list.index(key)`. What does
that cost, and why does it break the `O(1)` requirement? (You have met this exact cost
three times now: #103's `insert(0, x)`, #105's `.index()`, #622's `pop(0)`.)

**3.** So: a **dict from key to node**, and a **doubly linked list** holding the nodes
in use-order. The dict answers question 1's first sentence, and the ends of the list
answer the second. Now the question #707 made you trace: to unlink a node from the
middle of the list in `O(1)`, what do you need to already have? Say why the list must
be **doubly** linked, and check your answer against #707's question 5 - it is the same
argument, and this is what it was for.

**4.** Which operations count as a **use**? Trace the example: `capacity = 2`,
`put(1,1)`, `put(2,2)`, `get(1)`, `put(3,3)`. Which key is evicted - and which line
decided it? Write down the rule for all three of: `get` on a hit, `put` on a new key,
`put` on a key that is already there.

**5.** `put` on an **existing** key is the case people get wrong. It must update the
value **and** count as a use - and it must **not** evict anything, because the size did
not change. Write the branch, then say what a version that always evicts does to a
cache at exactly capacity.

**6.** `0 <= value`, so `0` is a legal value to store. And `get` returns `-1` for a
missing key. So what is wrong with each of these?

```python
if self.cache.get(key):      ...
if not self.cache.get(key):  return -1
```

Name the rule in one sentence. (#379 and #622 both have a test for this; now you have a
third.)

**7.** How would you test it? `get` returns a value, so lookups are visible - but the
**recency order** is not. A cache that evicts the wrong key looks perfectly healthy
until you ask for the key it should have kept, which might be fifty calls later. And
there is a nastier problem: you cannot inspect the order by calling `get`, because
`get` **is a use** and changes the thing you are measuring. What can you do instead?

## Two routes

**A - a dict plus your own doubly linked list** *(this is the assignment)*

```
self.cache = {}                       # key -> node
head, tail = Node(), Node()           # two sentinels, wired to each other
head.next, tail.prev = tail, head     # #707 route B, exactly
```

Keep **most recently used next to `head`** and **least recently used next to
`tail`**, so eviction is always `tail.prev`. Two private helpers do all the work and
you write the pointer surgery once:

- `_remove(node)` - `node.prev.next = node.next; node.next.prev = node.prev`. Two
  lines, **zero** `None` checks, because the sentinels guarantee every real node has a
  neighbour on both sides. That is what the two dummies bought you.
- `_insert_front(node)` - four assignments, also with no branches.

Then `get` is: miss -> `-1`; hit -> `_remove`, `_insert_front`, return value. And `put`
is: existing -> update, `_remove`, `_insert_front`; new -> insert, and if oversized,
evict `tail.prev` from both the list **and** the dict. That last "and" is the bug you
will actually write: removing the node but leaving the key in the dict, which makes a
later `get` return a node that is no longer in the list.

Both operations are `O(1)` - genuinely, not amortised.

**B - `collections.OrderedDict`** *(five lines, and worth understanding)*

`OrderedDict` has `move_to_end(key)` and `popitem(last=False)`, which are exactly
"mark as used" and "evict the oldest". `get` becomes three lines and `put` four, and
LeetCode accepts it.

Be honest about what it is: `OrderedDict` **is** a dict plus a doubly linked list, in
C, inside CPython. You are not avoiding the design, you are importing someone else's.
Which is the right call in production and the wrong call today - the same sentence
#707 used about wrapping a Python list.

> **One structure is not enough.** That is the whole lesson, and it is the first time
> in this repo that it has been true. A dict knows *where things are* but not *when
> they were touched*; a linked list knows *order* but cannot find anything. Bolt them
> together - the dict's values **are** the list's nodes - and both questions become
> `O(1)`. Almost every interesting data structure is two boring ones wired together
> like this.

In [1]:
class Node:
    def __init__(self, key=0, value=0):
        self.key = key
        self.value = value
        self.prev = None
        self.next = None


class LRUCache:

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}
        self.left = Node()
        self.right = Node()

        self.left.next = self.right
        self.right.prev = self.left

    def remove(self, node):
        prev = node.prev
        nxt = node.next

        prev.next = nxt
        nxt.prev = prev

    def insert(self, node):
        prev = self.right.prev
        nxt = self.right

        prev.next = node
        node.prev = prev

        node.next = nxt
        nxt.prev = node

    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        node = self.cache[key]
        self.remove(node)
        self.insert(node)

        return node.value

    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            self.remove(self.cache[key])
        node = Node(key, value)
        self.cache[key] = node
        self.insert(node)
        if len(self.cache) > self.capacity:
            lru = self.left.next

            self.remove(lru)
            del self.cache[lru.key]

### The test harness

Question 7's answer, and it has a genuine limitation worth understanding.

`check` replays a call sequence against your class **and** against a deliberately slow
model: a plain dict for the values plus a list of keys in use-order, where "mark as
used" is `list.remove` followed by `append`. That model is `O(n)` per operation, which
is exactly what you are not allowed to write - and exactly what you want in an oracle,
because it is obviously correct.

**What it cannot do, and why.** It cannot inspect your recency order between calls,
because the only tool available is `get`, and `get` *is* a use - measuring would change
what is being measured. (#362's harness hit the same wall for the same reason.) So this
harness compares return values only, and the test cases below are built so that a wrong
eviction surfaces within a call or two of the mistake: every case that evicts something
asks for the evicted key **and** the key that should have survived, immediately
afterwards.

`stress` does the rest of the work: thousands of random calls at tiny capacities, where
almost every `put` evicts and any error in the ordering shows up within a few calls.
Run this cell; don't edit it.

In [2]:
import random


def check(capacity, ops):
    '''Replay (op, args) against LRUCache and a slow-but-obvious list+dict model.'''
    log = [f"LRUCache({capacity})"]
    try:
        lru = LRUCache(capacity)
    except Exception as e:
        return False, [f"   !! LRUCache({capacity}) raised {type(e).__name__}: {e}"]

    data, order = {}, []            # order: least recently used first

    def use(k):
        if k in order:
            order.remove(k)
        order.append(k)

    for op, args in ops:
        call = f"{op}({', '.join(map(str, args))})"
        try:
            if op == "get":
                k = args[0]
                want = data.get(k, -1)
                if k in data:
                    use(k)
                got = lru.get(k)
                log.append(f"{call} -> {got!r}   (want {want})   order {order}")
                if got != want:
                    log.append(f"   !! {call} must return {want}, got {got!r}")
                    log.append(f"      cache should hold {dict(sorted(data.items()))}")
                    log.append(f"      use-order (oldest first) {order}")
                    return False, log
            else:
                k, v = args
                lru.put(k, v)
                if k in data:
                    data[k] = v
                    use(k)
                else:
                    if len(data) >= capacity:
                        oldest = order.pop(0)
                        del data[oldest]
                        log.append(f"{call}            evicts {oldest}")
                    data[k] = v
                    order.append(k)
                    if len(log) and not log[-1].endswith(f"evicts {k}"):
                        log.append(f"{call}            order {order}")
        except Exception as e:
            log.append(f"   !! {call} raised {type(e).__name__}: {e}")
            return False, log

    return True, log


def stress(n, capacity, seed=0, keys=8):
    '''Tiny capacity, small key space - almost every put evicts.'''
    random.seed(seed)
    ops = []
    for _ in range(n):
        k = random.randint(0, keys)
        if random.random() < 0.5:
            ops.append(("get", (k,)))
        else:
            ops.append(("put", (k, random.randint(0, 100))))
    return check(capacity, ops)


def report(name, ok, log, tail=6):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        for line in log[-tail:]:
            print(f"       {line}")

In [3]:
# tests
CASES = [
    ("the LeetCode example", 2, [
        ("put", (1, 1)), ("put", (2, 2)), ("get", (1,)), ("put", (3, 3)),
        ("get", (2,)), ("put", (4, 4)), ("get", (1,)), ("get", (3,)), ("get", (4,))]),

    ("get on an empty cache", 2, [("get", (1,)), ("get", (0,))]),

    ("capacity 1 - every put evicts", 1, [
        ("put", (1, 1)), ("get", (1,)), ("put", (2, 2)), ("get", (1,)), ("get", (2,))]),

    ("question 4: get counts as a use", 2, [
        ("put", (1, 1)), ("put", (2, 2)), ("get", (1,)), ("put", (3, 3)),
        ("get", (2,)),                      # -1: 2 was the oldest, it went
        ("get", (1,))]),                    # 1:  the get saved it

    ("question 5: put on an existing key updates and does NOT evict", 2, [
        ("put", (1, 1)), ("put", (2, 2)), ("put", (1, 10)),
        ("get", (1,)),                      # 10 - the value was updated
        ("get", (2,))]),                    # 2  - still there, nothing was evicted

    ("question 5: put on an existing key is also a use", 2, [
        ("put", (1, 1)), ("put", (2, 2)), ("put", (1, 10)), ("put", (3, 3)),
        ("get", (2,)),                      # -1: 2 became the oldest
        ("get", (1,)), ("get", (3,))]),

    ("question 6: value 0 is legal and is NOT a miss", 2, [
        ("put", (1, 0)), ("get", (1,)),     # 0, not -1
        ("put", (2, 0)), ("get", (2,)), ("get", (1,))]),

    ("question 6: key 0 is legal too", 2, [
        ("put", (0, 5)), ("get", (0,)), ("put", (1, 6)), ("put", (2, 7)),
        ("get", (0,))]),                    # -1: key 0 was evicted like any other

    ("the eviction victim is asked for immediately", 3, [
        ("put", (1, 1)), ("put", (2, 2)), ("put", (3, 3)),
        ("get", (1,)),                      # use 1, so 2 is now oldest
        ("put", (4, 4)),
        ("get", (2,)), ("get", (1,)), ("get", (3,)), ("get", (4,))]),

    ("re-putting an evicted key brings it back", 2, [
        ("put", (1, 1)), ("put", (2, 2)), ("put", (3, 3)),
        ("get", (1,)), ("put", (1, 100)), ("get", (1,)), ("get", (2,))]),

    ("a full cache read over and over never evicts", 3,
     [("put", (i, i)) for i in range(3)] + [("get", (i % 3,)) for i in range(30)]),

    ("capacity larger than anything we store", 100,
     [("put", (i, i * 2)) for i in range(20)] + [("get", (i,)) for i in range(20)]),
]

for name, cap, ops in CASES:
    report(name, *check(cap, ops))

for n, cap, seed, keys in [(200, 1, 1, 3), (500, 2, 2, 5), (2000, 3, 3, 8), (5000, 10, 4, 20)]:
    report(f"stress: {n} calls (capacity {cap}, seed {seed}, keys 0..{keys})",
           *stress(n, cap, seed, keys))

print("\ntrace of the LeetCode example:")
for line in check(2, CASES[0][2])[1]:
    print("  " + line)

OK   the LeetCode example
OK   get on an empty cache
OK   capacity 1 - every put evicts
OK   question 4: get counts as a use
OK   question 5: put on an existing key updates and does NOT evict
OK   question 5: put on an existing key is also a use
OK   question 6: value 0 is legal and is NOT a miss
OK   question 6: key 0 is legal too
OK   the eviction victim is asked for immediately
OK   re-putting an evicted key brings it back
OK   a full cache read over and over never evicts
OK   capacity larger than anything we store
OK   stress: 200 calls (capacity 1, seed 1, keys 0..3)
OK   stress: 500 calls (capacity 2, seed 2, keys 0..5)
OK   stress: 2000 calls (capacity 3, seed 3, keys 0..8)
OK   stress: 5000 calls (capacity 10, seed 4, keys 0..20)

trace of the LeetCode example:
  LRUCache(2)
  put(1, 1)            order [1]
  put(2, 2)            order [1, 2]
  get(1) -> 1   (want 1)   order [2, 1]
  put(3, 3)            evicts 2
  put(3, 3)            order [1, 3]
  get(2) -> -1   (want -1)   

## After it passes

- **Count the `None` checks in `_remove`.** There should be zero. Now delete the two
  sentinels, rewrite it, and count again - that difference is what #707's route B was
  teaching, and this is the problem where it pays.
- **Break it the way you were going to anyway.** In the eviction branch, remove the node
  from the list but *forget* `del self.cache[key]`. Run the tests. The failure is a
  `get` returning a value for a key that was evicted - find which case catches it, and
  note that the dict and the list must be updated together or not at all. That pairing
  is the only real invariant in this class.
- **The invariant list.** Write it out: every key in the dict points at a node that is
  in the list; the list holds exactly as many nodes as the dict has keys; that number
  never exceeds `capacity`; `head.next` is the most recently used and `tail.prev` the
  least. Then say which of the four each of `get` and `put` can break.
- **Build route B**, run the same tests, and `timeit` both at 200 000 operations. The
  `OrderedDict` version will win comfortably, because its linked list is C and yours is
  Python. Then say why writing yours was still the right call today.
- **Then change the eviction policy and watch the design hold.** LFU - evict the *least
  frequently* used - is #460, and it needs a count per node plus a structure keyed by
  count. FIFO needs no `get`-side bookkeeping at all. Write down which parts of your
  class survive each change; the parts that survive are the ones you designed well.
- **Make it real.** Add a maximum **age** as well as a maximum size, and you have a TTL
  cache - which means every node now carries a timestamp and you are back in #359's
  territory. Add `stats()` returning hits and misses, and you can finally answer the
  only question anyone asks about a cache: is it working?
- Siblings: **#707 Design Linked List** (route B is the list you built there, and this is
  what it was for), #460 LFU Cache (the hard sequel), #1472 Design Browser History (the
  same list, different question), #380 Insert Delete GetRandom O(1) (another problem
  solved by bolting two structures together).